# Praktikum Pertemuan 10 — Customer Churn

**Nama Lengkap:** Krisna Naufal Azhar Suhendar  
**NIM:** 240401070506  
**Kelas:** IF 405

## 1. Deskripsi

Customer churn merupakan kondisi ketika pelanggan berhenti menggunakan layanan suatu perusahaan.
Pada praktikum ini digunakan dataset Telco Customer Churn untuk membangun model klasifikasi
yang dapat memprediksi apakah seorang pelanggan berpotensi melakukan churn atau tidak.

Model yang digunakan adalah Random Forest Classifier dengan `class_weight="balanced"` untuk
mengatasi ketidakseimbangan kelas pada data churn.

#Kode Import Library

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

#Membaca Dataset

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Ukuran dataset:", df.shape)
display(df.head())

Ukuran dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#Kode Eksplorasi Data

In [10]:
print("Ukuran dataset:", df.shape)

print("\nTipe data:")
print(df.dtypes)

print("\nDistribusi kelas Churn:")
print(df["Churn"].value_counts())

print("\nProporsi kelas Churn:")
print(df["Churn"].value_counts(normalize=True))

Ukuran dataset: (7043, 21)

Tipe data:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Distribusi kelas Churn:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Proporsi kelas Churn:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


Prepocessing

In [11]:
# Membuat salinan dataset
data = df.copy()

# Menghapus customerID karena hanya merupakan identitas pelanggan
data = data.drop(columns=["customerID"])

# Mengubah TotalCharges menjadi numerik
data["TotalCharges"] = pd.to_numeric(
    data["TotalCharges"],
    errors="coerce"
)

# Menghapus baris dengan nilai kosong
data = data.dropna()

# Mengubah target Churn menjadi angka
data["Churn"] = data["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Ukuran data setelah preprocessing:", data.shape)

Ukuran data setelah preprocessing: (7032, 20)


Encoding

In [12]:
# Memisahkan fitur dan target
X = data.drop(columns=["Churn"])
y = data["Churn"]

# Encoding fitur kategorikal
X = pd.get_dummies(X, drop_first=True)

print("Jumlah fitur setelah encoding:", X.shape[1])
print("Jumlah data:", X.shape[0])

Jumlah fitur setelah encoding: 30
Jumlah data: 7032


#Train/Test Split

In [13]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Data training:", X_tr.shape)
print("Data testing :", X_te.shape)

print("\nProporsi Churn pada data training:")
print(y_tr.value_counts(normalize=True))

print("\nProporsi Churn pada data testing:")
print(y_te.value_counts(normalize=True))

Data training: (5625, 30)
Data testing : (1407, 30)

Proporsi Churn pada data training:
Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Proporsi Churn pada data testing:
Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64


#Random Forest

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

print("Random Forest berhasil dilatih.")

Random Forest berhasil dilatih.


#Evaluasi Model

In [15]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi kelas
y_pred = rf.predict(X_te)

# Prediksi probabilitas churn
y_prob = rf.predict_proba(X_te)[:, 1]

# Classification report
print("=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_te,
        y_pred,
        target_names=["Tidak Churn", "Churn"]
    )
)

# ROC-AUC
roc_auc = roc_auc_score(y_te, y_prob)

print("ROC-AUC:", round(roc_auc, 4))

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

 Tidak Churn       0.83      0.90      0.86      1033
       Churn       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC: 0.8199


#Prediksi Probabilitas Churn

In [16]:
# Membuat tabel hasil prediksi
hasil_prediksi = pd.DataFrame({
    "Actual_Churn": y_te.values,
    "Predicted_Churn": y_pred,
    "Churn_Probability": y_prob
})

# Urutkan dari probabilitas churn tertinggi
hasil_prediksi = hasil_prediksi.sort_values(
    by="Churn_Probability",
    ascending=False
)

print("10 pelanggan dengan probabilitas churn tertinggi:")

display(hasil_prediksi.head(10))

10 pelanggan dengan probabilitas churn tertinggi:


,Actual_Churn,Predicted_Churn,Churn_Probability
369,1,1,1.000000
1220,1,1,0.996667
728,0,1,0.996667
107,0,1,0.996667
304,1,1,0.996667
31,1,1,0.990000
261,1,1,0.983333
591,1,1,0.970000
1149,1,1,0.963333
446,0,1,0.960000


#Prediksi laporan churn yang mudah dibaca

In [17]:
hasil_prediksi["Churn_Probability"] = (
    hasil_prediksi["Churn_Probability"] * 100
).round(2)

display(hasil_prediksi.head(10))

,Actual_Churn,Predicted_Churn,Churn_Probability
369,1,1,100.00
1220,1,1,99.67
728,0,1,99.67
107,0,1,99.67
304,1,1,99.67
31,1,1,99.00
261,1,1,98.33
591,1,1,97.00
1149,1,1,96.33
446,0,1,96.00


## Kesimpulan

Pada praktikum ini telah dilakukan prediksi Customer Churn menggunakan algoritma Random Forest. Dataset Telco Customer Churn memiliki ketidakseimbangan kelas antara pelanggan yang melakukan churn dan tidak melakukan churn, sehingga digunakan parameter `class_weight="balanced"` untuk membantu model menangani kondisi tersebut.

Berdasarkan hasil evaluasi menggunakan precision, recall, F1-score, dan ROC-AUC, model dapat digunakan untuk membedakan pelanggan yang berpotensi churn dan pelanggan yang tidak churn. Nilai recall pada kelas Churn menjadi salah satu perhatian penting karena menunjukkan kemampuan model dalam menemukan pelanggan yang benar-benar berisiko berhenti menggunakan layanan.

Model juga menghasilkan probabilitas churn untuk setiap pelanggan. Pelanggan dengan nilai probabilitas churn yang tinggi dapat diprioritaskan oleh perusahaan untuk mendapatkan tindakan retensi, seperti pemberian penawaran atau layanan khusus.

Keterbatasan praktikum ini adalah model yang digunakan masih menggunakan parameter dasar dan belum dilakukan hyperparameter tuning atau perbandingan dengan algoritma klasifikasi lainnya. Selain itu, hasil prediksi model tidak dapat menjamin pelanggan pasti melakukan churn karena keputusan pelanggan juga dipengaruhi oleh faktor lain yang tidak terdapat dalam dataset.